# Adapt the baseline to a processed cell table

**Mapping template · cytokinechallenge.org**

This is a small-table mapping template, not a release-specific CD4 loader. No release format, official metadata field, or download location is assumed.

Download `cytokine_baseline.py` into the same folder as this notebook. The template uses only Python's standard library. Start with a **small, already processed CSV export** whose rows are cells and whose gene columns share a fixed expression scale. Large sparse matrices need a more scalable implementation; do not densify an entire single-cell dataset just to use this example.

Before export, select one comparable cell population and experimental context. Decide how dose, duration, batch, and other context define a condition. If those vary, grouping only by cytokine names is not sufficient.

Define the split before fitting learned preprocessing or choosing features. Do not select genes or tune a model using final test results. Review the release's actual schema, QC guidance, and license when available.


## 1. Map your columns explicitly

Fill in every `None` value, choose your gene columns, and specify at least one held-out combination. There are deliberately no assumed release column names. Set `CONTROL_LABEL` to the exact value your table uses for control, including `""` if appropriate.

Your condition field must contain either the control label or signal names separated by `SIGNAL_SEPARATOR`. Edit the parser if your encoding differs. Signal names must not themselves contain that separator.

This template evaluates unseen combinations within observed donors. Every tested donor requires a matching training control and training observations for each individual signal. The metric is illustrative mean-profile RMSE, not official scoring.


In [ ]:
from pathlib import Path
import csv
import importlib.util
import sys

CSV_PATH = None               # e.g. Path("my_small_processed_export.csv")
CELL_ID_COLUMN = None         # exact column name in your CSV
DONOR_COLUMN = None           # exact donor column name
CONDITION_COLUMN = None       # exact condition column name
CONTROL_LABEL = None          # exact control label, possibly ""
SIGNAL_SEPARATOR = ";"       # your chosen encoding, not an official format
GENE_COLUMNS = []             # explicit expression feature columns
HELD_OUT_COMBINATIONS = []    # e.g. [("your_signal_1", "your_signal_2")]


## 2. Load the companion implementation and define the mapping

The loader validates unique cell IDs, finite numeric expression, aligned genes, and explicit metadata columns. It does not normalize raw counts, select genes, infer controls, or harmonize experimental contexts.


In [ ]:
def load_companion(path=Path("cytokine_baseline.py")):
    if not path.is_file():
        raise FileNotFoundError("Download cytokine_baseline.py next to this notebook first.")
    spec = importlib.util.spec_from_file_location("cytokine_baseline", path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = module
    spec.loader.exec_module(module)
    return module


def load_processed_cells(path, cell_col, donor_col, condition_col,
                         control_label, separator, gene_cols):
    if not separator or not gene_cols or len(set(gene_cols)) != len(gene_cols):
        raise ValueError("Use a nonempty separator and a unique explicit gene list.")
    metadata = {cell_col, donor_col, condition_col}
    if len(metadata) != 3 or metadata.intersection(gene_cols):
        raise ValueError("Metadata columns must be distinct and separate from gene columns.")
    rows = []
    with Path(path).open(newline="", encoding="utf-8-sig") as handle:
        reader = csv.DictReader(handle)
        required = metadata.union(gene_cols)
        if reader.fieldnames is None or required - set(reader.fieldnames):
            raise ValueError("Missing required columns: %s" % sorted(required - set(reader.fieldnames or [])))
        for record in reader:
            raw_condition = record[condition_col].strip()
            signals = () if raw_condition == control_label.strip() else tuple(
                signal.strip() for signal in raw_condition.split(separator)
            )
            rows.append({"cell_id": record[cell_col].strip(),
                         "donor": record[donor_col].strip(),
                         "cytokines": signals,
                         "expression": {gene: float(record[gene]) for gene in gene_cols}})
    return rows


## 3. Run only after your mapping is complete

All cells can be run safely with the placeholders unchanged; the template prints a reminder and performs no loading. Once configured, every cell belonging to a held-out combination is excluded from training across every donor. If a required single signal or control is missing, prediction fails with an explanatory error rather than silently substituting another group.


In [ ]:
configured = (
    all(value is not None for value in [CSV_PATH, CELL_ID_COLUMN, DONOR_COLUMN,
                                        CONDITION_COLUMN, CONTROL_LABEL])
    and bool(GENE_COLUMNS) and bool(HELD_OUT_COMBINATIONS)
)

if not configured:
    print("Fill in the mapping, gene list, and held-out combinations in section 1.")
else:
    baseline = load_companion()
    rows = load_processed_cells(CSV_PATH, CELL_ID_COLUMN, DONOR_COLUMN,
                                CONDITION_COLUMN, CONTROL_LABEL,
                                SIGNAL_SEPARATOR, GENE_COLUMNS)
    baseline.validate_rows(rows)
    train, test = baseline.split_by_condition(rows, HELD_OUT_COMBINATIONS)
    print("Training cells:", len(train), "Test cells:", len(test))
    print("Training donors:", sorted({row["donor"] for row in train}))
    model = baseline.fit_additive(train)
    result = baseline.evaluate(model, test)
    print("Additive mean-profile RMSE:", result["rmse_additive"])
    print("Control-only mean-profile RMSE:", result["rmse_control_only"])


## 4. Review the result in context

- Save your actual mapping, selected population, preprocessing choices, gene list, and split alongside any report.
- Inspect donor-condition coverage and per-group errors before reporting a single average.
- Separate validation used for model choices from the final test set; do not repeatedly optimize against final test performance.
- Means can hide distinct cell states. Add distributional evaluation if your scientific target or model output requires it.
- This baseline is additive on your chosen expression scale. Changing that scale changes the assumption and the metric.
- Do not interpret cell-level sampling as independent donor replication.

The full release may need different grouping, train/test restrictions, or scoring. Follow its documented benchmark protocol once available.
